# Demo: Gravitational Wave Spectrum

This notebook demonstrates how to compute and compare the predicted stochastic gravitational wave background from the dark $U(1)$ FOPT using `GWFromParams`.

**Pipeline step:** Final layer — takes the macroscopic FOPT parameters $(T_n, \alpha, \beta/H)$ from `demo_FOPT_params.ipynb` and produces the GW power spectrum $h^2\Omega_{\rm GW}(f)$.

**Modules used:** `src/RGE/GW_RGE_spectrum.py`

**Three GW sources are included:**
- Bubble wall collisions (scalar field gradient energy)
- Sound waves in the plasma
- MHD turbulence

**Contents:**
1. Spectrum from a single set of FOPT parameters
2. Comparison of multiple loop orders / renormalization schemes vs NANOGrav-15yr
3. Renormalization-scale uncertainty bands

In [ ]:
%load_ext autoreload
%autoreload 2

%run ../startup.py

from scipy.stats import gaussian_kde

In [ ]:
# Instantiate the GW spectrum calculator
gw = gw_spec.GWFromParams()

# Frequency array covering the PTA and LISA bands
f = np.logspace(-11, -6, 200)

## 1. Spectrum from a single set of FOPT parameters

Load the output of `demo_FOPT_params.ipynb` and compute the full GW spectrum for one representative $g_D$ value.

`gw.spectra(Tn, alpha, betaH)` returns four callables:
`(h2_total, h2_bubble, h2_sw, h2_turb)` as functions of frequency $f$.

In [ ]:
# Load parameter scan from demo_FOPT_params output
try:
    fopt_data = np.loadtxt("../data/final_data/gD_Tn_alpha_beta_RGE_piT_HT_all.dat")
    gD_scan, Tn_scan, alpha_scan, beta_scan = fopt_data.T
    print(f"Loaded {len(gD_scan)} parameter points")
except FileNotFoundError:
    print("Run demo_FOPT_params.ipynb first to generate the parameter file.")
    print("Using hard-coded example values instead.")
    # Example values for gD = 0.8 (high-T, mu = pi)
    gD_scan   = np.array([0.8])
    Tn_scan   = np.array([0.118])
    alpha_scan = np.array([5.6])
    beta_scan  = np.array([133.8])

In [ ]:
# Pick one gD point close to 0.8
idx_single = np.argmin(np.abs(gD_scan - 0.8))
Tn_ex, alpha_ex, beta_ex = Tn_scan[idx_single], alpha_scan[idx_single], beta_scan[idx_single]
print(f"gD = {gD_scan[idx_single]:.3f},  Tn = {Tn_ex:.5f},  alpha = {alpha_ex:.3e},  beta = {beta_ex:.2f}")

h2_tot, h2_bub, h2_sw, h2_turb = gw.spectra(Tn_ex, alpha_ex, beta_ex, kappa_col=1.0)

fig, ax = new_figure(figsize=(7, 5), dpi=120)
fig.patch.set_facecolor('white')

ax.plot(f, [h2_tot(fi)  for fi in f], color='black',     lw=2.2, label='Total')
ax.plot(f, [h2_bub(fi)  for fi in f], color='royalblue', lw=1.4, ls='--', label='Bubbles')
ax.plot(f, [h2_sw(fi)   for fi in f], color='darkorange', lw=1.4, ls='--', label='Sound waves')
ax.plot(f, [h2_turb(fi) for fi in f], color='limegreen',  lw=1.4, ls='--', label='Turbulence')

ax.text(0.05, 0.92, fr'$g_D={gD_scan[idx_single]:.2f}$, $T_n={Tn_ex:.4f}$',
        transform=ax.transAxes, fontsize=11)

ps.apply_standard_formatting(
    ax,
    xlog=True, ylog=True,
    xlabel=r'$f\,[{\rm Hz}]$',
    ylabel=r'$h^2\Omega_{\rm GW}(f)$',
    ylim=(1e-18, 1e-6),
    xlim=(1e-11, 1e-6),
)
ax.legend(loc='upper left', frameon=False, fontsize=11)
plt.tight_layout()

## 2. Multiple loop orders vs NANOGrav-15yr

Compare the GW spectra from different treatments of the effective potential (4D HT daisy, 1-loop LO/NLO, 2-loop NLO) against the NANOGrav-15yr posterior. The parameters below are obtained from separate runs of `demo_FOPT_params` with different perturbative schemes, all at $g_D = 0.6$.

In [ ]:
# Load NANOGrav-15yr posterior samples
data_ng = np.loadtxt("../data/other_data/NG15.dat")
logfreqs_ng = data_ng[:, 0]
freqs_ng    = 10**logfreqs_ng
samples_ng  = 10**data_ng[:, 1:]   # convert log10 to linear

In [ ]:
# FOPT parameters for gD = 0.6 under different perturbative treatments
# Source: separate runs of demo_FOPT_params with different Veff options
spectra_params = {
    "OPA":         (0.004925878072664542,   327835.9,  32.19),
    "4D HT (Daisy)": (0.009736573968270185,  106058.6,  42.46),
    "1-L (LO)":    (0.008657363356312823,   169679.0,  41.08),
    "1-L (NLO)":   (0.012347489336953803,    41006.8,  44.38),
    "2-L (NLO)": (0.009398665817136863,   122153.5,  43.15),
    "2-L (Mixed)":  (0.009314813138090460,   126611.8,  43.05),
}

plot_styles = {
    "OPA":            {"color": "orange",     "lw": 3,   "ls": ":" },
    "4D HT (Daisy)":  {"color": "limegreen",  "lw": 2.0, "ls": "-" },
    "1-L (LO)":       {"marker": r"$\ast$",   "color": "royalblue",  "s": 80},
    "1-L (NLO)":      {"marker": "d",         "edgecolor": "navy",   "facecolor": "none", "lw": 1.0, "s": 65},
    "2-L (NLO)":    {"marker": "s",         "edgecolor": "#8B1E3F", "facecolor": "none", "lw": 1.0, "s": 65},
    "2-L (Mixed)":    {"marker": "o",         "edgecolor": "#C99E10", "facecolor": "none", "lw": 1,   "s": 80},
}

In [ ]:
# Build GW spectra
spectra_funcs = {}
for key, (Tn_k, alpha_k, beta_k) in spectra_params.items():
    h2_tot_k, *_ = gw.spectra(Tn_k, alpha_k, beta_k, kappa_col=1.0)
    spectra_funcs[key] = h2_tot_k

# --- Plot ---
fig, ax = plt.subplots(figsize=(7, 5.5), dpi=120)
fig.patch.set_facecolor('white')

# NANOGrav-15yr violin distributions
for i in range(len(freqs_ng)):
    y = samples_ng[i]
    y = y[y > 0]
    if len(y) < 10:
        continue
    kde  = gaussian_kde(np.log10(y))
    yv   = np.linspace(np.log10(y.min()), np.log10(y.max()), 200)
    dens = kde(yv)
    w    = freqs_ng[i] * 0.12
    dens_scaled = dens / dens.max() * w
    ax.fill_betweenx(10**yv,
                     freqs_ng[i] - dens_scaled,
                     freqs_ng[i] + dens_scaled,
                     alpha=0.12, color='red', edgecolor='none', zorder=1)

# GW curves
for key, h2func in spectra_funcs.items():
    style = plot_styles[key]
    y_vals = np.array([h2func(fi) for fi in f])
    if "marker" in style:
        ax.scatter(f, y_vals, label=key,
                   marker=style["marker"],
                   s=style.get("s", 35),
                   edgecolor=style.get("edgecolor", None),
                   facecolor=style.get("facecolor", None),
                   lw=style.get("lw", 0.1),
                   color=style.get("color", None),
                   zorder=3)
    else:
        ax.plot(f, y_vals, label=key,
                color=style.get("color", None),
                lw=style.get("lw", 2.0),
                ls=style.get("ls", "-"),
                zorder=3)

ax.text(1e-10, 1e-13, r'$g_D = 0.6$', fontsize=19, ha='left', va='bottom')
ax.text(2.1e-8, 1e-13, 'NANOGrav-15yr', fontsize=11, ha='left', va='bottom',
        rotation=90, color='darkred')

ps.apply_standard_formatting(
    ax,
    xlog=True, ylog=True,
    xlabel=r'$f\,[{\rm Hz}]$',
    ylabel=r'$h^2\Omega_{\rm GW}(f)$',
    ylim=(1e-14, 1e-6),
    xlim=(1e-11, 1e-6),
)
ax.legend(loc='upper left', frameon=False, scatterpoints=1, numpoints=1, fontsize=12, ncol=1)
plt.tight_layout()
plt.savefig("../plots/GW_comparison_loop_orders.pdf", dpi=300, bbox_inches='tight', facecolor='white')
print("Saved: plots/GW_comparison_loop_orders.pdf")

## 3. Renormalization-scale uncertainty bands

For the high-$T$ (daisy) and 2-loop (NLO) treatments, we vary $\mu$ over $[\pi/4 T,\, \pi T,\, 4\pi T]$ to estimate the perturbative uncertainty. The spread of the GW spectrum quantifies residual scale dependence.

In [ ]:
# HT band: mu = 0.25piT, piT, 4piT
Tn_HT,  aHT,  bHT  = 0.009736573968270185, 106058.6, 42.46    # mu = pi T
Tn_HT1, aHT1, bHT1 = 0.012692660003153515, 36724.9,  45.14    # mu = 0.25 pi T
Tn_HT2, aHT2, bHT2 = 0.007768203124059500, 261751.,  39.98    # mu = 4 pi T

# 2-loop (NLO) band
Tn_2L,  a_2L,  b_2L  = 0.009398665817136863, 122153.5,  43.15   # central
Tn_2L1, a_2L1, b_2L1 = 0.009107556658922054, 138536.0,  43.06   # mu = 0.25 mu_match
Tn_2L2, a_2L2, b_2L2 = 0.009298674143090531, 127493.0,  43.21   # mu = 4 mu_match

# OPA reference
Tn_OPA, a_OPA, b_OPA = 0.004925878072664542, 327835.9, 32.19

In [ ]:
# Build all spectra
h2_HT,  *_ = gw.spectra(Tn_HT,  aHT,  bHT,  kappa_col=1.0)
h2_HT1, *_ = gw.spectra(Tn_HT1, aHT1, bHT1, kappa_col=1.0)
h2_HT2, *_ = gw.spectra(Tn_HT2, aHT2, bHT2, kappa_col=1.0)
h2_2L,  *_ = gw.spectra(Tn_2L,  a_2L,  b_2L,  kappa_col=1.0)
h2_2L1, *_ = gw.spectra(Tn_2L1, a_2L1, b_2L1, kappa_col=1.0)
h2_2L2, *_ = gw.spectra(Tn_2L2, a_2L2, b_2L2, kappa_col=1.0)
h2_OPA, *_ = gw.spectra(Tn_OPA, a_OPA, b_OPA, kappa_col=1.0)

def eval_spec(h2func):
    y = np.array([h2func(fi) for fi in f])
    return y, np.isfinite(y) & (y > 0)

y_HT,  m_HT  = eval_spec(h2_HT)
y_HT1, m_HT1 = eval_spec(h2_HT1)
y_HT2, m_HT2 = eval_spec(h2_HT2)
y_2L,  m_2L  = eval_spec(h2_2L)
y_2L1, m_2L1 = eval_spec(h2_2L1)
y_2L2, m_2L2 = eval_spec(h2_2L2)
y_OPA, m_OPA = eval_spec(h2_OPA)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5), dpi=120)
fig.patch.set_facecolor('white')

# NANOGrav-15yr violins
for i in range(len(freqs_ng)):
    y = samples_ng[i]
    y = y[y > 0]
    if len(y) < 10:
        continue
    kde  = gaussian_kde(np.log10(y))
    yv   = np.linspace(np.log10(y.min()), np.log10(y.max()), 200)
    dens = kde(yv)
    w    = freqs_ng[i] * 0.12
    dens_scaled = dens / dens.max() * w
    ax.fill_betweenx(10**yv,
                     freqs_ng[i] - dens_scaled,
                     freqs_ng[i] + dens_scaled,
                     alpha=0.12, color='red', edgecolor='none', zorder=1)

# Central lines
ax.plot(f[m_HT],  y_HT[m_HT],  color='limegreen', ls=':', lw=2.5, label=r'4D HT, $\mu=\pi T$')
ax.plot(f[m_2L],  y_2L[m_2L],  color='purple',    ls=':', lw=2.5, label=r'2-L (NLO), $\mu_{\rm match}$')
ax.plot(f[m_OPA], y_OPA[m_OPA], color='orange',   ls=':',  lw=2.5, label='OPA')

# Uncertainty bands (min/max over the three scale choices)
m_band_HT = m_HT1 & m_HT2
ax.fill_between(f[m_band_HT],
                np.minimum(y_HT1, y_HT2)[m_band_HT],
                np.maximum(y_HT1, y_HT2)[m_band_HT],
                color='limegreen', alpha=0.2, edgecolor='none', zorder=2,
                label=r'4D HT band ($\mu/4T$ to $4\mu T$)')

m_band_2L = m_2L1 & m_2L2
ax.fill_between(f[m_band_2L],
                np.minimum(y_2L1, y_2L2)[m_band_2L],
                np.maximum(y_2L1, y_2L2)[m_band_2L],
                color='purple', alpha=0.2, edgecolor='none', zorder=2,
                label=r'2-L (NLO) band')

ax.text(1e-10, 1e-13, r'$g_D = 0.6$', fontsize=19, ha='left', va='bottom')
ax.text(2.1e-8, 1e-13, 'NANOGrav-15yr', fontsize=11, ha='left', va='bottom',
        rotation=90, color='darkred')

ps.apply_standard_formatting(
    ax,
    xlog=True, ylog=True,
    xlabel=r'$f\,[{\rm Hz}]$',
    ylabel=r'$h^2\Omega_{\rm GW}(f)$',
    ylim=(1e-14, 1e-6),
    xlim=(1e-11, 1e-6),
)
ax.legend(loc='upper left', frameon=False, fontsize=11, ncol=1)
plt.tight_layout()
plt.savefig("../plots/GW_scale_uncertainty.pdf", dpi=300, bbox_inches='tight', facecolor='white')
print("Saved: plots/GW_scale_uncertainty.pdf")